<a href="https://colab.research.google.com/github/ionutbirescu/lyrics-data-mining/blob/main/GeniusLyrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Are Lyrics Getting Simpler and More Negative? A Data-Driven check
Ionuț Birescu, Cristina Ioja

Nush daca sa pastram asta sa aratam ca am incercat cu API

In [ ]:
!pip install lyricsgenius

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.3 MB/s eta 0:00:00


In [ ]:
!pip install -q playwright
!playwright install chromium
!playwright install-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 20.8 MB/s eta 0:00:00
175.4 MiB [] 0% 0.0s175.4 MiB [] 0% 56.9s175.4 MiB [] 0% 25.0s175.4 MiB [] 0% 20.0s175.4 MiB [] 0% 20.2s175.4 MiB [] 0% 14.2s175.4 MiB [] 1% 6.9s175.4 MiB [] 1% 7.5s175.4 MiB [] 1% 9.0s175.4 MiB [] 1% 10.1s175.4 MiB [] 1% 10.9s175.4 MiB [] 1% 11.5s175.4 MiB [] 2% 8.4s175.4 MiB [] 3% 6.7s175.4 MiB [] 4% 6.0s175.4 MiB [] 5% 5.5s175.4 MiB [] 5% 5.0s175.4 MiB [] 6% 4.4s175.4 MiB [] 7% 4.5s175.4 MiB [] 8% 4.1s175.4 MiB [] 9% 4.2s175.4 MiB [] 9% 4.4s175.4 MiB [] 9% 4.5s175.4 MiB [] 9% 4.9s175.4 MiB [] 12% 3.9s175.4 MiB [] 13% 3.6s175.4 MiB [] 14% 3.4s175.4 MiB [] 15% 3.1s175.4 MiB [] 17% 2.9s175.4 MiB [] 18% 2.8s175.4 MiB [] 19% 2.7s175.4 MiB [] 20% 2.7s175.4 MiB [] 20% 2.8s175.4 MiB [] 20% 2.9s175.4 MiB [] 22% 2.8s175.4 MiB [] 23% 2.6s175.4 MiB [] 25% 2.4s175.4 MiB [] 26% 2.3s175.4 MiB [] 28% 2.1s175.4 MiB [] 30% 2.0s175.4 MiB [] 32% 1.8s175.4 MiB [] 34% 1.7s175.4 MiB [] 36% 1.6s175.4 MiB [] 38% 1.5s175.4 MiB [] 

In [ ]:
import requests

GENIUS_TOKEN = "sD9aBWxiYECun7eODG0-VYlo9t9b03dLfLSGQhaxHeSDKuNH8mDsc4foiyLO36kc"
BASE_URL = "https://api.genius.com"

headers = {"Authorization": f"Bearer {GENIUS_TOKEN}"}

from difflib import SequenceMatcher

def best_match(query, hits):
    def score(hit):
        return SequenceMatcher(None, query.lower(), hit["result"]["full_title"].lower()).ratio()
    return max(hits, key=score)

def search_genius(query):
    search_url = f"{BASE_URL}/search"
    data = {"q": query}
    response = requests.get(search_url, params=data, headers=headers)
    response.raise_for_status()
    return response.json()["response"]["hits"]

results = search_genius("Livin' on a Prayer")
top = best_match("Living on a Prayer", results)
print(top["result"]["full_title"], top["result"]["url"])

Livin' on a Prayer by Bon Jovi https://genius.com/Bon-jovi-livin-on-a-prayer-lyrics


Link for lyricsgenius: https://github.com/johnwmillr/LyricsGenius

In [ ]:
import requests, asyncio, csv, os
import pandas as pd
from playwright.async_api import async_playwright

GENIUS_TOKEN = "sD9aBWxiYECun7eODG0-VYlo9t9b03dLfLSGQhaxHeSDKuNH8mDsc4foiyLO36kc"
HEADERS = {"Authorization": f"Bearer {GENIUS_TOKEN}"}
OUT = "lyrics_raw.csv"
FIELDS = ["search_artist","artist", "title", "lyrics", "genre", "year", "decade"]
SONGS_PER_ARTIST = 12
DELAY = 6                  # seconds between lyric-page fetches
# The page gives lyrics but NOT reliable genre, so the dict is the actual ground truth.
ARTISTS = {
    ("pop",     "1980s"): ["Madonna", "Michael Jackson", "Whitney Houston"],
    ("pop",     "2010s"): ["Taylor Swift", "Ed Sheeran", "Ariana Grande"],
    ("rock",    "1980s"): ["Bon Jovi", "U2", "Queen"],
    ("rock",    "2010s"): ["Imagine Dragons", "Arctic Monkeys", "Foo Fighters"],
    ("rap",     "1990s"): ["2Pac", "The Notorious B.I.G.", "Nas"],
    ("rap",     "2010s"): ["Drake", "Kendrick Lamar", "J. Cole"],
    ("pop",     "2020s"): ["Sombr", "Olivia Rodrigo", "Billie Eilish"],
    ("rock",    "2020s"): ["Måneskin", "Imagine Dragons", "Arctic Monkeys"],
    ("rap",     "2020s"): ["Kendrick Lamar", "Drake", "Travis Scott"],
    ("rb",      "2020s"): ["The Weeknd", "SZA", "Doja Cat"],
}

#We find an artist's ID because we need it to search songs
def get_artist_id(name):
    r = requests.get("https://api.genius.com/search",
                     params={"q": name}, headers=HEADERS)
    hits = r.json()["response"]["hits"]
    for h in hits:
        if name.lower() in h["result"]["primary_artist"]["name"].lower():
            return h["result"]["primary_artist"]["id"]
    return hits[0]["result"]["primary_artist"]["id"] if hits else None

#we take the found ID and return a list of up to n of the artist's songs
def get_artist_songs(artist_id, n):
    songs, page = [], 1
    while len(songs) < n and page <= 5:
        r = requests.get(f"https://api.genius.com/artists/{artist_id}/songs",
                         params={"per_page": 20, "page": page, "sort": "popularity"},
                         headers=HEADERS)
        batch = r.json()["response"]["songs"]
        if not batch:
            break
        for s in batch:
            songs.append({"title": s["title"],
                          "url": s["url"],
                         "artist_names": s.get("artist_names", "")}) #we include even remixes/songs where the artist is a feature
        page += 1
    return songs[:n]

# Playwright: fetch lyric text from one page
async def fetch_lyrics(page, url):
    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_selector('div[data-lyrics-container="true"]', timeout=15000)
    containers = await page.query_selector_all('div[data-lyrics-container="true"]')
    return "\n".join([await c.inner_text() for c in containers])

# we never re-scrape a song we already have
def load_done(path):
    if os.path.exists(path) and os.path.getsize(path) > 0:   # skip empty files
        try:
            df = pd.read_csv(path)
            return set(zip(df["artist"], df["title"]))
        except pd.errors.EmptyDataError:                     # file exists but has no header yet
            return set()
    return set()

# main loop
async def scrape():
    done = load_done(OUT)
    new_file = not os.path.exists(OUT)
    f = open(OUT, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    if new_file:
        writer.writeheader()

    async with async_playwright() as p:
        browser = await p.chromium.launch()
        # a realistic UA helps a little against bot detection
        ctx = await browser.new_context(
            user_agent=("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) "
                        "Chrome/124.0 Safari/537.36"))
        page = await ctx.new_page()

        for (genre, decade), artists in ARTISTS.items():
            for artist in artists:
                aid = get_artist_id(artist)
                if not aid:
                    print("no artist id:", artist); continue
                for song in get_artist_songs(aid, SONGS_PER_ARTIST):
                    key = (artist, song["title"])
                    if key in done:
                        continue
                    try:
                        lyrics = await fetch_lyrics(page, song["url"])
                        if lyrics and len(lyrics) > 100:   # filter empty/snippet pages
                            writer.writerow({
                                "search_artist": artist,                       # "Madonna" (your bucket key)
                                "artist": song["artist_names"] or artist,
                                "title": song["title"],
                                "lyrics": lyrics, "genre": genre,
                                "year": "", "decade": decade})
                            f.flush()                      # <-- save immediately
                            done.add(key)
                            print("ok  ", artist, "—", song["title"])
                        else:
                            print("thin", artist, "—", song["title"])
                    except Exception as e:
                        print("FAIL", artist, "—", song["title"], "|", type(e).__name__)
                    await asyncio.sleep(DELAY)
        await browser.close()
    f.close()
    print("\nDONE. Total rows:", len(load_done(OUT)))

await scrape()

FAIL Madonna — EVIL J0RDAN | TimeoutError


##KAGGLE Dataset (this is where the project begins)

In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()   # select your kaggle.json here

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"ionubirescu","key":"d7e9e37d3b80ba779db46b3c1be88bb5"}'}

In [ ]:
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d carlosgdcj/genius-song-lyrics-with-language-information
!unzip -q genius-song-lyrics-with-language-information.zip
!ls -lh *.csv

Dataset URL: https://www.kaggle.com/datasets/carlosgdcj/genius-song-lyrics-with-language-information
License(s): unknown
100% 3.04G/3.04G [00:23<00:00, 137MB/s]

-rw-r--r-- 1 root root  22M Jun 21 19:31 lyrics_clean.csv
-rw-r--r-- 1 root root   53 Jun 21 19:27 lyrics_raw.csv
-rw-r--r-- 1 root root 8.5G Jan 11  2023 song_lyrics.csv


###Preprocessing

We start from a massive lyrics file - 9 GB - so we read it bit by bit and keep only English songs, from 4 genres (pop, rock, rap, R&B) released between 1980 and 2023. Each song gets labeled with its decade so we can track how things change over time. Since some decades may have more songs that others, we trim each group down to 200 so we get a balanced set for lyrics_raw.csv to work with from here on.

In [ ]:
import pandas as pd

PATH = "song_lyrics.csv"          # change if Cell 2's ls showed a different name
GENRES = ["pop", "rock", "rap", "rb"]
PER_CELL = 200                    # songs per genre x decade

frames = []
for chunk in pd.read_csv(PATH, chunksize=200_000,
                         usecols=["title", "artist", "tag", "year", "language", "lyrics"]):
    chunk = chunk[
        (chunk["language"] == "en") &
        (chunk["tag"].isin(GENRES)) &
        (chunk["year"].between(1980, 2025))
    ].copy()
    chunk["decade"] = (chunk["year"] // 10 * 10).astype(int).astype(str) + "s"
    frames.append(chunk)

df = pd.concat(frames, ignore_index=True)

# balance: cap each genre x decade cell so no group dominates
df = (df.groupby(["tag", "decade"], group_keys=False)
        .apply(lambda g: g.sample(min(len(g), PER_CELL), random_state=42)))

df = df.rename(columns={"tag": "genre"})[
    ["artist", "title", "lyrics", "genre", "year", "decade"]
].reset_index(drop=True)

df.to_csv("lyrics_raw.csv", index=False)
print("shape:", df.shape)
print(df.groupby(["genre", "decade"]).size())

The dataset came mostly pre-cleaned so we only strip: Verge/Chorus tags, multispaces, non-words etc. We then create two versions of each song - a light one that keeps negations like "not"/"never" for sentiment and a heavier one with stopwords removed and words lemmatized for TF-IDF, the model and search.

In [ ]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

_SECTION_TAG  = re.compile(r"\[.*?\]")
_WEIRD_SPACES = re.compile(r"[\u2005\u200b\u00a0\u2009\u202f]")
_MULTISPACE   = re.compile(r"[ \t]+")
_MULTINEWLINE = re.compile(r"\n{2,}")
_NON_WORD     = re.compile(r"[^a-z'\s]")

def clean_text(raw):
    if not isinstance(raw, str):
        return ""
    t = _EMBED_TAIL.sub(" ", raw.strip())
    t = _SECTION_TAG.sub(" ", t)
    t = _WEIRD_SPACES.sub(" ", t)
    t = _MULTISPACE.sub(" ", t)
    t = _MULTINEWLINE.sub("\n", t)
    return t.strip().lower()

def tokenize(clean):
    """Heavy pipeline: drop punctuation/digits -> remove stopwords -> lemmatize. For TF-IDF / model / IR."""
    if not clean:
        return []
    out = []
    for w in _NON_WORD.sub(" ", clean).split():
        w = w.strip("'")
        if len(w) < 3 or w in STOPWORDS:
            continue
        out.append(LEMMATIZER.lemmatize(w))
    return out

def preprocess_dataframe(df, text_col="lyrics", min_words=20):
    """Add clean_text, tokens, tokens_str, word_count, unique_word_count, ttr; drop too-short/duplicate rows."""
    df = df.copy()
    df["clean_text"]        = df[text_col].apply(clean_text)
    df["tokens"]            = df["clean_text"].apply(tokenize)
    df["tokens_str"]        = df["tokens"].apply(lambda toks: " ".join(toks))
    df["word_count"]        = df["tokens"].apply(len)
    df["unique_word_count"] = df["tokens"].apply(lambda toks: len(set(toks)))
    df["ttr"]               = df.apply(
        lambda r: r["unique_word_count"] / r["word_count"] if r["word_count"] else 0.0, axis=1)

    before = len(df)
    df = df[df["word_count"] >= min_words]
    df = df.drop_duplicates(subset="clean_text").reset_index(drop=True)
    print(f"[preprocess] kept {len(df)}/{before} rows ({before - len(df)} dropped: too-short/duplicate)")
    return df

df = pd.read_csv("lyrics_raw.csv")
df = preprocess_dataframe(df)
df.to_csv("lyrics_clean.csv", index=False)
print("shape:", df.shape)
df.head(3)

In [ ]:
from google.colab import files
files.download("lyrics_clean.csv")

###TF-IDF